# Importer les bibliotheques

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Importer les donnees

In [56]:
data = pd.read_csv('evmar.csv')

In [57]:
data.head()

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire
0,9/22/2017,NaN,Autres,NaN,MADAGASCAR,NaN,NaN,NaN,NaN,44.05,-27.1,NaN,NaN,NaN,"Le Commandant russe du navire MT ETC MENA, pav...",NaN
1,12/27/2017,NaN,Autres,NaN,CANAL DE MOZAMBIQUE,NaN,NaN,NaN,NaN,47.008206,-12.865347,NaN,NaN,NaN,"A la position 12°52S et 047°E, le FV GIBELLE, ...",NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2/10/2017,NaN,Evènement naturel,NaN,Océan Indien,NaN,NaN,NaN,NaN,56.544268,-20.731773,NaN,NaN,NaN,cyclone tropical CARLOS : fortes precipitation...,NaN
4,2/15/2017,NaN,Evènement naturel,NaN,Canal du Mozambique,NaN,NaN,NaN,NaN,36.271156,-19.11166,NaN,NaN,NaN,cyclone DINEO : fortes precipitations et vents...,NaN


In [58]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1930 entries, 0 to 1929
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Date debut         1862 non-null   object
 1   Date fin           22 non-null     object
 2   Thématique         1862 non-null   object
 3   Objets             1617 non-null   object
 4   District           1711 non-null   object
 5   Commune            100 non-null    object
 6   Localite           871 non-null    object
 7   Region             28 non-null     object
 8   Types              194 non-null    object
 9   Longitude          1513 non-null   object
 10  Latitude           1513 non-null   object
 11  Personne concerne  7 non-null      object
 12  Mort               4 non-null      object
 13  Colonne1           14 non-null     object
 14  description        1813 non-null   object
 15  Commentaire        503 non-null    object
dtypes: object(16)
memory usage: 241.4+ KB


In [59]:
data.describe()

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire
count,1862,22,1862,1617,1711,100,871,28,194,1513,1513,7,4,14,1813,503
unique,969,19,54,1595,240,90,516,14,88,1299,1293,6,4,9,1811,503
top,04/12/20,4/10/2019,Autres,Restriction sur le trafic maritime,MADAGASCAR,Ambatolaoka,Port,ANALANJIROFO,SAR,"49°24'59.78""E","18° 9'22.31""S",1 mort,La population côtière du Fokontany d’Andrevo s...,naufrage,Le satellite de l’EMSA (CLEANSEANET) a détecté...,Nosy Be est une île riche en biodiversité mais...
freq,18,2,654,3,276,3,43,6,13,31,31,2,1,4,2,1


# Preprocessing des données

## Objectifs du preprocessing :

1. **Standardisation des dates** : 
   - Uniformiser toutes les dates au format `dd/mm/yyyy`
   - Gérer les différents formats de dates présents dans le dataset

2. **Filtrage thématique** : 
   - Sélectionner uniquement les données relatives aux incidents maritimes
   - Nettoyer les variations dans les libellés de thématiques

3. **Filtrage temporel** : 
   - Conserver uniquement les données de la période 2017-2022
   - Trier les données par ordre chronologique (date de début)

4. **Validation et nettoyage** :
   - Vérifier la cohérence des données filtrées
   - Supprimer les doublons éventuels

### 1- Uniformiser les dates de la forme dd/mm/year

In [60]:
# Creer une copie de la data pour eviter de modifier l'original
data_copy = data.copy()

In [ ]:
# Fonction pour standardiser les dates
def standardize_date(date_str):
    """
    Standardise les dates au format dd/mm/yyyy
    Gère plusieurs formats d'entrée et retourne None pour les dates invalides
    """
    if pd.isna(date_str):
        return None

    date_str = str(date_str).strip()
    
    # Si c'est déjà vide ou invalide
    if not date_str or date_str.lower() in ['nan', 'none', 'null', '']:
        return None

    # Liste des formats possibles à tester
    formats = [
        '%d/%m/%Y',    # 01/12/2020
        '%d/%m/%y',    # 01/12/20
        '%Y-%m-%d',    # 2020-12-01
        '%d-%m-%Y',    # 01-12-2020
        '%d-%m-%y',    # 01-12-20
        '%d.%m.%Y',    # 01.12.2020
        '%d.%m.%y',    # 01.12.20
        '%Y/%m/%d',    # 2020/12/01
    ]

    for fmt in formats:
        try:
            date_obj = pd.to_datetime(date_str, format=fmt, errors='raise')
            # Retourner au format dd/mm/yyyy
            return date_obj.strftime('%d/%m/%Y')
        except:
            continue

    # Essai avec parser automatique en dernier recours
    try:
        date_obj = pd.to_datetime(date_str, errors='coerce', dayfirst=True)
        if pd.notna(date_obj):
            return date_obj.strftime('%d/%m/%Y')
    except:
        pass

    # Si aucun format ne fonctionne, retourner None
    print(f"Date non reconnue: {date_str}")
    return None

Application de la standardisation des dates avec validation

In [72]:
print("=== STANDARDISATION DES DATES ===")

print("==================")
print("Statistiques avant traitement")
print(f"Nombre total de lignes: {len(data_copy)}")
print(f"Dates de début non nulles avant: {data_copy['Date debut'].notna().sum()}")
print(f"Dates de fin non nulles avant: {data_copy['Date fin'].notna().sum()}")

print("\n==================")
print("Standardiser les dates de début")
date_debut_standardized = data_copy['Date debut'].apply(standardize_date)
data_copy['Date debut'] = date_debut_standardized

print("\n==================")
print("Standardiser les dates de fin")
date_fin_standardized = data_copy['Date fin'].apply(standardize_date)
data_copy['Date fin'] = date_fin_standardized

print("\n==================")
print("Statistiques après traitement")
print(f"\nDates de début non nulles après: {data_copy['Date debut'].notna().sum()}")
print(f"Dates de fin non nulles après: {data_copy['Date fin'].notna().sum()}")

print("\nAffichage d'exemples de dates standardisées")
print(f"\nExemples de dates standardisées:")
valid_dates = data_copy[data_copy['Date debut'].notna()]['Date debut'].head(5)
for i, date in enumerate(valid_dates, 1):
    print(f"  {i}. {date}")

print("✓ Standardisation des dates terminée")

=== STANDARDISATION DES DATES ===
Statistiques avant traitement
Nombre total de lignes: 1930
Dates de début non nulles avant: 1862
Dates de fin non nulles avant: 22

Standardiser les dates de début

Standardiser les dates de fin

Statistiques après traitement

Dates de début non nulles après: 1862
Dates de fin non nulles après: 22

Affichage d'exemples de dates standardisées

Exemples de dates standardisées:
  1. 22/09/2017
  2. 27/12/2017
  3. 02/10/2017
  4. 15/02/2017
  5. 03/10/2017
✓ Standardisation des dates terminée

Standardiser les dates de fin

Statistiques après traitement

Dates de début non nulles après: 1862
Dates de fin non nulles après: 22

Affichage d'exemples de dates standardisées

Exemples de dates standardisées:
  1. 22/09/2017
  2. 27/12/2017
  3. 02/10/2017
  4. 15/02/2017
  5. 03/10/2017
✓ Standardisation des dates terminée


In [73]:
data_copy.head()

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire,Thematique_norm
0,22/09/2017,None,Autres,NaN,MADAGASCAR,NaN,NaN,NaN,NaN,44.05,-27.1,NaN,NaN,NaN,"Le Commandant russe du navire MT ETC MENA, pav...",NaN,autres
1,27/12/2017,None,Autres,NaN,CANAL DE MOZAMBIQUE,NaN,NaN,NaN,NaN,47.008206,-12.865347,NaN,NaN,NaN,"A la position 12°52S et 047°E, le FV GIBELLE, ...",NaN,autres
2,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02/10/2017,None,Evènement naturel,NaN,Océan Indien,NaN,NaN,NaN,NaN,56.544268,-20.731773,NaN,NaN,NaN,cyclone tropical CARLOS : fortes precipitation...,NaN,evènement naturel
4,15/02/2017,None,Evènement naturel,NaN,Canal du Mozambique,NaN,NaN,NaN,NaN,36.271156,-19.11166,NaN,NaN,NaN,cyclone DINEO : fortes precipitations et vents...,NaN,evènement naturel


### 2- Trier seulement les donnees de incident maritime

In [74]:
# Examiner les valeurs uniques dans la colonne Thématique
print("Valeurs uniques dans 'Thématique':")
print(data_copy['Thématique'].unique())
print(f"\nNombre total de lignes: {len(data_copy)}")
print(f"Nombre de valeurs non nulles dans 'Thématique': {data_copy['Thématique'].notna().sum()}")

Valeurs uniques dans 'Thématique':
['Autres' nan 'Evènement naturel' 'Evènement naturel maritime/ AHSC'
 'Evènement naturel maritime/ AHSC ' 'Incidents maritimes'
 'Incidents Maritimes' 'Incidents Maritimes ' 'Infrastructure critique'
 'Pêche Illégale, Non Reportée et Non Règlementée (INN)'
 'Trafic d’armes, de drogues et contrebande'
 'Trafic d’armes, de drogues et contrebande ' 'Trafics et contrebandes'
 'VAPM' 'Migration illégale & Trafic d’être humain par voie maritime'
 'Trafic et contrebande par voie maritime' 'Environnement Marin'
 'Evenement naturel maritime' 'Incident maritime'
 'Infrastructure critique maritime'
 "Migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime"
 "Migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime "
 'Migration illégale & trafic d’être humain par voie maritime'
 'Pêche irrégulière non reportée et non reglémentée'
 'Pêche illégale, Non reporté et 

In [75]:
def filter_maritime_incidents(df):
    """
    Filtre les données pour ne conserver que les incidents maritimes
    Utilise plusieurs critères de recherche pour capturer toutes les variantes
    """
    print("=== FILTRAGE DES INCIDENTS MARITIMES ===")
    
    # Normalisation de la colonne thématique
    df['Thematique_norm'] = df['Thématique'].str.lower().str.strip()
    
    # Affichage des thématiques uniques pour diagnostic
    print("Thématiques présentes dans les données:")
    thematiques_uniques = df['Thematique_norm'].dropna().unique()
    for i, theme in enumerate(sorted(thematiques_uniques), 1):
        print(f"  {i}. {theme}")
    
    # Critères de recherche pour les incidents maritimes
    patterns_maritime = [
        'incident.*maritime',
        'maritime.*incident',
        'accident.*maritime', 
        'maritime.*accident',
        'incident.*mer',
        'mer.*incident',
        'navire',
        'bateau',
        'embarcation'
    ]
    
    # Création du masque de filtrage
    mask = pd.Series([False] * len(df))
    
    for pattern in patterns_maritime:
        pattern_mask = df['Thematique_norm'].str.contains(pattern, na=False, regex=True)
        mask = mask | pattern_mask
        matches = pattern_mask.sum()
        if matches > 0:
            print(f"Pattern '{pattern}': {matches} correspondances")
    
    # Application du filtre
    maritime_incidents = df[mask].copy()
    
    print(f"\nRésultat du filtrage:")
    print(f"  - Données originales: {len(df)} lignes")
    print(f"  - Incidents maritimes: {len(maritime_incidents)} lignes")
    print(f"  - Pourcentage conservé: {len(maritime_incidents)/len(df)*100:.1f}%")
    
    # Nettoyage de la colonne temporaire
    maritime_incidents = maritime_incidents.drop('Thematique_norm', axis=1)
    
    return maritime_incidents

In [76]:
# Application du filtrage des incidents maritimes
incident_data = filter_maritime_incidents(data_copy)

# Vérification rapide des résultats
print(f"\n=== APERÇU DES DONNÉES FILTRÉES ===")
if len(incident_data) > 0:
    print("Colonnes disponibles:")
    for i, col in enumerate(incident_data.columns, 1):
        print(f"  {i}. {col}")
    
    print(f"\nPremières thématiques dans les données filtrées:")
    thematiques_filtrees = incident_data['Thématique'].dropna().unique()[:5]
    for i, theme in enumerate(thematiques_filtrees, 1):
        print(f"  {i}. {theme}")
else:
    print("⚠️ Aucun incident maritime trouvé dans les données")

=== FILTRAGE DES INCIDENTS MARITIMES ===
Thématiques présentes dans les données:
  1. acte violent en mer
  2. actes violents en mer
  3. attaque violent en mer (vam)
  4. autres
  5. environnement marin
  6. evenement naturel maritime
  7. evènement naturel
  8. evènement naturel maritime
  9. evènement naturel maritime (ahsc)
  10. evènement naturel maritime/ ahsc
  11. evénement naturel maritime
  12. illegal, unreported & unregulated fishing (iuu)
  13. incident maritime
  14. incidents maritimes
  15. infrastructure critique
  16. infrastructure critique maritime
  17. infrastructures
  18. irregular migration and illicit trafficking/of migrants by sea
  19. migration illégale & trafic d’être humain par voie maritime
  20. migration irrégulière et trafics d'être humain par voie maritime et trafic d’être humain par voie maritime
  21. plaisance et tourisme maritime
  22. plaisance/tourisme maritime
  23. pêche illégale non reportée et non règlementée (inn)
  24. pêche illégale non 

In [77]:
incident_data.head()

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire
8,02/10/2017,None,Incidents maritimes,NaN,Katsepy Mahajanga/ Madagascar,NaN,NaN,NaN,NaN,46.279045,-15.746659,NaN,NaN,chavirement,chavirement d'une embarcation : mort d'un tour...,NaN
9,02/11/2017,None,Incidents maritimes,NaN,Ambodiatafana Madagascar,NaN,NaN,NaN,NaN,49.467173,-18.167217,NaN,NaN,noyade,noyade d'un jeune étudiant 11 Fév,NaN
10,25/02/2017,None,Incidents maritimes,NaN,Katsepy Mahajanga/ Madagascar,NaN,NaN,NaN,NaN,46.289816,-15.737907,NaN,NaN,naufrage,Le bac Makuba a coulé 25 Fév,NaN
11,17/03/2017,None,Incidents Maritimes,NaN,madagascar,NaN,NaN,NaN,NaN,43.872715,-18.042964,NaN,NaN,naufrage,Un boutre qui a quitté Morondava pour aller à...,NaN
12,31/03/2017,None,Incidents Maritimes,NaN,OCEAN INDIEN,NaN,NaN,NaN,NaN,68.761516,-20.827302,NaN,NaN,décès à bord,Un marin philippin âgé de 48 ans a été découve...,NaN


### 3- filtrer par date debut

In [78]:
# Filtrage et tri par période 2017-2022
print("=== FILTRAGE ET TRI PAR PÉRIODE (2017-2022) ===")

if len(incident_data) == 0:
    print("⚠️ Aucune donnée d'incident maritime à traiter")
else:
    # Conversion des dates pour le tri et filtrage
    print("Conversion des dates de début...")
    incident_data['Date_debut_dt'] = pd.to_datetime(
        incident_data['Date debut'], 
        format='%d/%m/%Y', 
        errors='coerce'
    )
    
    # Statistiques sur les dates
    dates_valides = incident_data['Date_debut_dt'].notna().sum()
    dates_invalides = incident_data['Date_debut_dt'].isna().sum()
    
    print(f"Dates valides: {dates_valides}")
    print(f"Dates invalides: {dates_invalides}")
    
    if dates_valides > 0:
        print(f"Période des données: {incident_data['Date_debut_dt'].min().strftime('%d/%m/%Y')} à {incident_data['Date_debut_dt'].max().strftime('%d/%m/%Y')}")
    
    # Définition de la période cible
    start_date = pd.to_datetime('01/01/2017', format='%d/%m/%Y')
    end_date = pd.to_datetime('31/12/2022', format='%d/%m/%Y')
    
    print(f"\nPériode de filtrage: {start_date.strftime('%d/%m/%Y')} à {end_date.strftime('%d/%m/%Y')}")
    
    # Application du filtre temporel
    mask_periode = (
        (incident_data['Date_debut_dt'] >= start_date) & 
        (incident_data['Date_debut_dt'] <= end_date)
    )
    
    incident_data_filtered = incident_data[mask_periode].copy()
    
    # Tri par date croissante
    incident_data_sorted = incident_data_filtered.sort_values(
        by='Date_debut_dt', 
        ascending=True
    ).reset_index(drop=True)
    
    # Suppression de la colonne temporaire
    incident_data_sorted = incident_data_sorted.drop('Date_debut_dt', axis=1)
    
    # Statistiques finales
    print(f"\n=== RÉSULTATS DU FILTRAGE ===")
    print(f"Données avant filtrage temporel: {len(incident_data)} lignes")
    print(f"Données dans la période 2017-2022: {len(incident_data_sorted)} lignes")
    print(f"Pourcentage conservé: {len(incident_data_sorted)/len(incident_data)*100:.1f}%")
    
    if len(incident_data_sorted) > 0:
        print(f"Première date: {incident_data_sorted['Date debut'].iloc[0]}")
        print(f"Dernière date: {incident_data_sorted['Date debut'].iloc[-1]}")
        
        # Distribution par année
        incident_data_sorted['Annee'] = pd.to_datetime(
            incident_data_sorted['Date debut'], 
            format='%d/%m/%Y'
        ).dt.year
        
        distribution_annuelle = incident_data_sorted['Annee'].value_counts().sort_index()
        print(f"\nDistribution par année:")
        for annee, count in distribution_annuelle.items():
            print(f"  {annee}: {count} incidents")
        
        incident_data_sorted = incident_data_sorted.drop('Annee', axis=1)
    else:
        print("⚠️ Aucune donnée trouvée pour la période 2017-2022")

print("\n✓ Preprocessing terminé avec succès!")

=== FILTRAGE ET TRI PAR PÉRIODE (2017-2022) ===
Conversion des dates de début...
Dates valides: 269
Dates invalides: 0
Période des données: 05/02/2017 à 31/12/2022

Période de filtrage: 01/01/2017 à 31/12/2022

=== RÉSULTATS DU FILTRAGE ===
Données avant filtrage temporel: 269 lignes
Données dans la période 2017-2022: 269 lignes
Pourcentage conservé: 100.0%
Première date: 05/02/2017
Dernière date: 31/12/2022

Distribution par année:
  2017: 33 incidents
  2018: 20 incidents
  2019: 41 incidents
  2020: 49 incidents
  2021: 63 incidents
  2022: 63 incidents

✓ Preprocessing terminé avec succès!


In [79]:
# Sauvegarde des données traitées et résumé final
print("=== SAUVEGARDE ET RÉSUMÉ FINAL ===")

if 'incident_data_sorted' in locals() and len(incident_data_sorted) > 0:
    # Sauvegarde en CSV
    output_filename = 'incidents_maritimes_sorted_2017_2022.csv'
    incident_data_sorted.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"✓ Données sauvegardées dans: {output_filename}")
    
    # Résumé final
    print(f"\n=== RÉSUMÉ DU PREPROCESSING ===")
    print(f"Dataset original: {len(data)} lignes")
    print(f"Après filtrage incidents maritimes: {len(incident_data)} lignes")
    print(f"Après filtrage période 2017-2022: {len(incident_data_sorted)} lignes")
    print(f"Taux de conservation final: {len(incident_data_sorted)/len(data)*100:.2f}%")
    
    print(f"\nColonnes dans le dataset final:")
    for i, col in enumerate(incident_data_sorted.columns, 1):
        non_null = incident_data_sorted[col].notna().sum()
        print(f"  {i}. {col}: {non_null}/{len(incident_data_sorted)} valeurs non nulles")
    
    # Aperçu des données finales
    print(f"\n=== APERÇU DES DONNÉES FINALES ===")
    print("Premières lignes:")
    display(incident_data_sorted.head(3))
    
else:
    print("⚠️ Aucune donnée à sauvegarder")

print(f"\n{'='*50}")
print("PREPROCESSING TERMINÉ AVEC SUCCÈS!")
print(f"{'='*50}")

=== SAUVEGARDE ET RÉSUMÉ FINAL ===
✓ Données sauvegardées dans: incidents_maritimes_sorted_2017_2022.csv

=== RÉSUMÉ DU PREPROCESSING ===
Dataset original: 1930 lignes
Après filtrage incidents maritimes: 269 lignes
Après filtrage période 2017-2022: 269 lignes
Taux de conservation final: 13.94%

Colonnes dans le dataset final:
  1. Date debut: 269/269 valeurs non nulles
  2. Date fin: 0/269 valeurs non nulles
  3. Thématique: 269/269 valeurs non nulles
  4. Objets: 184/269 valeurs non nulles
  5. District: 210/269 valeurs non nulles
  6. Commune: 39/269 valeurs non nulles
  7. Localite: 130/269 valeurs non nulles
  8. Region: 20/269 valeurs non nulles
  9. Types: 73/269 valeurs non nulles
  10. Longitude: 263/269 valeurs non nulles
  11. Latitude: 263/269 valeurs non nulles
  12. Personne concerne: 4/269 valeurs non nulles
  13. Mort: 1/269 valeurs non nulles
  14. Colonne1: 14/269 valeurs non nulles
  15. description: 269/269 valeurs non nulles
  16. Commentaire: 66/269 valeurs non nul

,Date debut,Date fin,Thématique,Objets,District,Commune,Localite,Region,Types,Longitude,Latitude,Personne concerne,Mort,Colonne1,description,Commentaire
0,05/02/2017,None,Incidents Maritimes,NaN,MADAGASCAR,NaN,NaN,NaN,NaN,46.32589,-15.766938,NaN,NaN,NaN,"A Mahajanga, le cadavre d’un homme retrouvé au...",NaN
1,25/02/2017,None,Incidents maritimes,NaN,Katsepy Mahajanga/ Madagascar,NaN,NaN,NaN,NaN,46.289816,-15.737907,NaN,NaN,naufrage,Le bac Makuba a coulé 25 Fév,NaN
2,07/03/2017,None,Incidents Maritimes,NaN,OCEAN INDIEN,NaN,NaN,NaN,NaN,54.740555,-21.401068,NaN,NaN,assistance technique,"Le vraquier IRIS II (IMO 9286906, dwt 75798, c...",NaN



PREPROCESSING TERMINÉ AVEC SUCCÈS!


In [80]:
incident_data_sorted["description"]

0      A Mahajanga, le cadavre d’un homme retrouvé au...
1                           Le bac Makuba a coulé 25 Fév
2      Le vraquier IRIS II (IMO 9286906, dwt 75798, c...
3      Un boutre qui a quitté Morondava  pour aller à...
4      Un marin philippin âgé de 48 ans a été découve...
                             ...                        
264    Le 22 novembre 2022 à 10h du matin une coque b...
265    Le 27 novembre 2022 deux hommes âgés de 25 ans...
266    Le 12 janvier 2022 vers 16h les autorités de F...
267    Le 20 décembre 2022 à 14h30m le navire « SIREN...
268    Le 29 décembre 2022 un pêcheur de la Commune r...
Name: description, Length: 269, dtype: object